# Form 1065 Worksheet

## F1065 Services

Ingests a General Ledger for an LLC rental business and maps account balances
to the appropriate lines of IRS Form 1065 (U.S. Return of Partnership Income).

## Notes: General Ledger to Form 1065


## General Ledeger Account Mapping (rental LLC):
````
  Acct.Asset.Purchase    -> Schedule L (total assets / depreciable property)
  Acct.Cash.Expense      -> Page 1 Line 20 (other deductions)
  Acct.Cash.Income       -> Page 1 Line 1a (gross receipts / rents)
  Acct.Cash.Investment   -> Schedule L (partners' capital contributions)
  Acct.Cash.Misc         -> Page 1 Line 7 (other income)
  Acct.Cash.Util         -> Page 1 Line 20 (utilities, part of other deductions)
  Acct.Interest.Income   -> Page 1 Line 5 (interest income)
  Balance                -> Schedule L (ending cash/bank balance)
````





In [1]:
# Init Services
from __future__ import annotations
import os
from pathlib import Path

import math
from dataclasses import dataclass, field
from typing import Dict, Optional
import pandas as pd

from irs.Form1065  import Form1065Preparer



## IRS Worksheet for LLC - Form 1065, Sch K,L 

In [2]:
#Form 1065 Worksheet using LLC General ledger accounts
# ════════════════════════════════════════════════════════════════════════════

# General Ledger - All accounts
# LLC General Ledger
from ledger.LLC import LLC

top = os.path.join(Path.home(), 'GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group')
llc = LLC('WBGroupLLC',debug=False, top=top)
# Save entity information
eDict = llc.entity

# General Ledger Accounts Dict
glDict  = round(llc.acctsDF(),2).to_dict()

# Compute Beginning_cash amount based on earliest Investment from Asset DB
df = pd.DataFrame([a for a in llc.assets().load() if a['oID'][0] == 'i']).sort_values(by='dt')
sDT = sorted(df.dt.unique())[0]
begCash = df[df.dt == sDT].amt.sum()

preparer = Form1065Preparer(
    glDict,
    tax_year     = llc.yr,
    entity_name  = eDict['entity_name'],
    ein          = eDict['ein'],
    beginning_cash = begCash,
)

preparer.compute()
preparer.print_summary()


  IRS FORM 1065 WORKSHEET  |  Tax Year 2025
  W&B Group, LLC  |  EIN: 39-3842347

  PAGE 1 — INCOME
  Line 1a  Gross Receipts (Rental)      $      4,000.53
  Line 5   Interest Income               $        400.00
  Line 7   Other Income (Misc)           $         29.47
  ────────────────────────────────────────────────────
           TOTAL INCOME                  $      4,430.00

  PAGE 1 — DEDUCTIONS
  Line 20  Other Deductions
             Cash Expenses               $      1,766.92
             Utilities                   $      1,056.95
  ────────────────────────────────────────────────────
           TOTAL DEDUCTIONS              $      2,823.87

  LINE 22  ORDINARY INCOME / (LOSS)     $      1,606.13

  SCHEDULE K — DISTRIBUTIVE SHARE ITEMS
  K-1 Box 1  Ordinary Income/(Loss)     $      1,606.13
  K-1 Box 2  Net Rental Income           $      4,000.53
  K-1 Box 5  Interest Income             $        400.00
  K-1 Box 11 Other Income                $         29.47
  ─────────────

## Build Form 1065

In [3]:
# Form 1065 Constructor
from irs.pdfMap import irsGLMap

pm = irsGLMap(llc)

f1065Dict = pm.f1065Dict(glDict)

# ── Attempt to fill PDF if it exists ────────────────────────────────────
# Clean entity name -> File Basename, no space, '&'
llcBN = ''.join(llc.entity['entity_name'].replace('&','').split())

# PDF Files 
yeDIR = llc.acctDir(dirName = 'ye')
PDF_IN  = os.path.join(yeDIR, 'Forms_IRS', 'Form_1065-IRS.pdf')
PDF_OUT = os.path.join(yeDIR, f'Form_1065_{llc.yr}_{llcBN}.pdf')
WS_OUT = PDF_OUT.replace("Form_1065", 'GenLedger_WorkSheet')



# Construct Form 1065: cross reference general ledger x F1065 template

from irs.pdfFill import pdfFill

# Initialize Form 1065  - pdfKey
pf = pdfFill(PDF_IN, PDF_OUT, glDict = f1065Dict, verbose=True)
fDict = pf.fillPDF()




✅  Saved → 'Form_1065_2025_WBGroup,LLC.pdf'  (85 fields filled)


# Schedule K-1 - per Partner

In [4]:
# Build Schedule K per Partner

## Statement Exclusion of Schedule K


- [>> § 1.761–2 Exclusion -subchapter K ; Chap 1, IRS Code](https://www.govinfo.gov/content/pkg/CFR-2024-title26-vol10/pdf/CFR-2024-title26-vol10-sec1-761-2.pdf)
- [>> Election to Exclude Applicable Unincorporated Organizations from the Application of Subchapter K](https://www.regulations.gov/document/IRS-2024-0054-0001
- [>>](https://www.google.com/search?q=I+have+2+partners+in+an+llc+with+60%25+and+40%25+ownership%2C+how+should+scheudule+K+be+drafted&sca_esv=9a3fab58edaf9218&rlz=1C5CHFA_enUS1122US1122&sxsrf=ANbL-n4gsTJYulT5wU6vgLcDhfirNkUj_A%3A1773687667139&ei=c1O4aayRCJegqtsPhJDYuAs&biw=1086&bih=823&ved=0ahUKEwjslt66jaWTAxUXkGoFHQQIFrcQ4dUDCBE&uact=5&oq=I+have+2+partners+in+an+llc+with+60%25+and+40%25+ownership%2C+how+should+scheudule+K+be+drafted&gs_lp=Egxnd3Mtd2l6LXNlcnAiWUkgaGF2ZSAyIHBhcnRuZXJzIGluIGFuIGxsYyB3aXRoIDYwJSBhbmQgNDAlIG93bmVyc2hpcCwgaG93IHNob3VsZCBzY2hldWR1bGUgSyBiZSBkcmFmdGVkSIX2AVAAWJr0AXAFeAGQAQGYAcUCoAHTPqoBCTQwLjM0LjEuMbgBA8gBAPgBAZgCRKAC5TbCAgQQIxgnwgILEAAYgAQYkQIYigXCAgoQABiABBhDGIoFwgIWEC4YgAQYsQMY0QMYQxiDARjHARiKBcICEBAAGIAEGLEDGIMBGBQYhwLCAggQLhiABBixA8ICCxAuGIAEGLEDGIMBwgIOEC4YgAQYsQMYgwEYigXCAgUQABiABMICCxAAGIAEGLEDGIMBwgINEAAYgAQYsQMYFBiHAsICBxAAGIAEGArCAhMQLhiABBixAxjRAxiDARjHARgKwgIFEC4YgATCAggQABiABBixA8ICCxAuGIAEGLEDGIoFwgIKEAAYgAQYFBiHAsICBhAAGBYYHsICBRAhGJ8FwgIFECEYoAHCAgsQABiABBiGAxiKBcICBRAAGO8FwgIFECEYqwLCAggQABiABBiiBJgDAJIHBzI5LjM4LjGgB_CRA7IHBzI0LjM4LjG4B9Q2wgcJMy40Mi4yMi4xyAe8AYAIAA&sclient=gws-wiz-serp)

The IRS guide states the LLC should include a "document contains proposed regulations that would provide certain administrative requirements for unincorporated organizations taking advantage of modifications to the rules governing elections to be excluded from the application of partnership tax rules. These proposed regulations would affect unincorporated organizations and their members, including tax-exempt organizations, the District of Columbia, State and local governments, Indian Tribal governments, Alaska Native Corporations, the Tennessee Valley Authority, rural electric cooperatives, and certain agencies and instrumentalities. The proposed regulations would also update the procedure for obtaining permission to revoke a section 761(a) election."

#### Question 32
A qualifying syndicate, pool, joint venture, or similar organization may elect under section 761(a) not to be treated as a partnership
for federal income tax purposes and won’t be required to file Form 1065 except for the year of election. If an election out of
subchapter K is being made for the tax year, answer “Yes” and attach a statement that contains:

- The names, addresses, and identification numbers of all the
members of the organization;
- A statement that the organization qualifies under
Regulations section 1.761-2(a), paragraph (1), and either
paragraph (2) or (3);
- A statement that all members of the organization elect to
exclude the organization from all of subchapter K; and
- A statement indicating the availability of the agreement
under which the organization operates (or for an oral
agreement, from whom the provisions of the agreement may
be obtained).

For calendar-year organizations, Form 1065 must be filed by March 15 following the close of the first calendar year for which
the section 761(a) election is being made. The filing date for fiscal-year organizations is the 15th day of the 3rd month
following the close of the 1st fiscal year. These dates are subject to filing extensions. See Regulations section 1.761-2 for more information concerning making the election out of subchapter K

## Export Worksheet to PDF

- AccountingData/2025/YE_Tax Records/*PDF

In [5]:
# Export PDF
import json
from irs.Form1065  import Form1065Preparer

try:
    preparer.export_pdf(WS_OUT)
except ImportError as e:
    print(f"[PDF skipped] {e}")

# Export JSON 

data = preparer.to_dict()
print("\nJSON snapshot (page1 only):")
print(json.dumps(data["page1"], indent=2))

✅  PDF saved → GenLedger_WorkSheet_2025_WBGroup,LLC.pdf

JSON snapshot (page1 only):
{
  "line_1a_gross_receipts": 4000.53,
  "line_5_interest_income": 400.0,
  "line_7_other_income": 29.47,
  "total_income": 4430.000000000001,
  "line_20_other_deductions": 2823.87,
  "total_deductions": 2823.87,
  "ordinary_income_loss": 1606.130000000001
}


## How/Guide GL and Form1065 are produced

### Class Architecture

The module has 3 data classes + 1 main class + a PDF builder:

| Class | Name | Module | Desc | 
| ---- | ---- | ---- | ---- |
| Data | Form1065Preparer | irs.data_F1065 | Main class — ingest GL, compute, export |
| Data | - F1065Page1 | irs.data_F1065 | Income & deduction lines (Line 1a, 5, 7, 20, 22) |
| Data | - Schedule K | irs.data_SchK | Partners' distributive share (K-1 boxes) |
| Data | - Schedule L | irs.data_schL | Balance sheet (assets, capital, liabilities) |
| Code |  _build_pdf() | irs.pdf | Internal ReportLab PDF report |

## Usage:
````
  gl = {
      "Acct.Asset.Purchase":  -214113.95,
      "Acct.Cash.Expense":      -1766.92,
      "Acct.Cash.Income":        4000.53,
      "Acct.Cash.Investment":  219227.00,
      "Acct.Cash.Misc":            29.47,
      "Acct.Cash.Util":         -1056.95,
      "Acct.Interest.Income":     400.00,
      "Balance":                 6719.18,
  }
  preparer = Form1065Preparer(gl, tax_year=2024,
                               entity_name="Sunset Ridge Rentals LLC",
                               ein="12-3456789")
  preparer.compute()
  preparer.print_summary()
  preparer.export_pdf("Form_1065_Worksheet.pdf")
````

